# Start with a dictionary · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 01. Start with a dictionary

Read, access, and change named values in a Python dictionary.

Before you start: You only need to recognize text, numbers, and the print function.

Official reference: https://docs.python.org/3/tutorial/datastructures.html#dictionaries

### M01C01. What are the key and the value?

A dictionary stores pairs: a key and its value. The key is a name you use to find a value, such as "Name" or "Age". Curly brackets create the dictionary; a colon separates each key from its value. Here, one dictionary describes one fictional student.

**Read:** In "Age": 20, which part is the key and which part is the value?

Answer: "Age" is the key. The integer 20 is its value.

**Predict:** What will len(student) print: the number of letters, pairs, or students?

Answer: It prints 3, the number of key-value pairs.

**Explain:** Why is "20" different from 20?

Answer: "20" is text; 20 is an integer. Numbers stored as text need conversion before arithmetic.



In [ ]:
# Predict before running: What will len(student) print: the number of letters, pairs, or students?
# Text uses quotes. A whole number does not need quotes.
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
print(student)
print(type(student).__name__)
print(len(student))

**Your turn:** Create a dictionary for Bilal, age 21, at Dubai. Add a fourth key called Program with value IT. Print the dictionary and its number of pairs.

For the check, keep these result names: `student`.

**Hint:** Use four key: value pairs inside one pair of curly brackets.

**Watch for this:** Do not mix up braces { } for dictionaries with brackets [ ] used for lists and access.

In [ ]:
# Change this code to complete the task.
# Question: Create a dictionary for Bilal, age 21, at Dubai. Add a fourth key called Program with value IT. Print the dictionary and its number of pairs.
student = {"Name": "Bilal", "Age": 21, "Campus": "Dubai", "Program": "IT"}
print(student)
print(len(student))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert student['Name'] == 'Bilal' and student['Age'] == 21
assert student['Program'] == 'IT' and len(student) == 4
print("Code checks passed. Review the written explanation as well.")

The result has four pairs. Program is a new key, so the dictionary length increases to 4.

**Quick check:** What does the colon do in "Age": 20?

1. It ends the dictionary.
2. It connects a key to its value.
3. It converts text to a number.

Answer: 2. A colon separates a dictionary key from its associated value.

**Explain what you learned:** I can identify the keys, values, and number of pairs in a dictionary.

### M01C02. Find a value by its key

Square brackets after a dictionary look up a key. The key must match exactly, including upper and lower case. get() is useful when a key might be missing: you can supply a default value instead of raising KeyError.

**Read:** What does student["Name"] ask Python to find?

Answer: It asks for the value associated with the exact key "Name".

**Predict:** What will the three print statements display?

Answer: Amira, then 20, then Not provided, each on a separate line.

**Explain:** Would student["name"] work here? Why?

Answer: No. The key is "Name", with an uppercase N. The different key "name" is absent, so direct access raises KeyError.



In [ ]:
# Predict before running: What will the three print statements display?
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
print(student["Name"])
print(student["Age"])
print(student.get("Email", "Not provided"))

**Your turn:** Store the campus in campus. Use get() to request Program with the default "Undeclared" and save the value in program. Store whether "Age" is a key in has_age. Print all three values.

For the check, keep these result names: `campus`, `has_age`, `program`.

**Hint:** Use student["Campus"], student.get(...), and "Age" in student.

**Watch for this:** A default from get() does not insert the missing key into the dictionary.

In [ ]:
# Change this code to complete the task.
# Question: Store the campus in campus. Use get() to request Program with the default "Undeclared" and save the value in program. Store whether "Age" is a key in has_age. Print all three values.
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
campus = student["Campus"]
program = student.get("Program", "Undeclared")
has_age = "Age" in student
print(campus)
print(program)
print(has_age)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert campus == 'RAK' and program == 'Undeclared' and has_age is True
print("Code checks passed. Review the written explanation as well.")

Campus is RAK. Program is missing, so get() returns the chosen default. Membership with in checks keys, so has_age is True.

**Quick check:** Which expression safely returns "Unknown" when Phone is missing?

1. student["Phone", "Unknown"]
2. student["Phone"]
3. student.get("Phone", "Unknown")

Answer: 3. get(key, default) supplies the default only when the requested key is absent.

**Explain what you learned:** I can choose between direct key access and a safe lookup with a default.

### M01C03. Change, add, and remove a pair

Assigning a value to an existing key replaces its old value. Assigning to a new key adds a pair. pop() removes a key and returns its value. A dictionary cannot hold two separate entries with the same key.

**Read:** Which line changes an existing pair, and which line adds a new pair?

Answer: student["Age"] = 21 changes Age. student["Program"] = "AI" adds Program.

**Predict:** How many pairs remain after Campus is removed?

Answer: Three: Name, Age, and Program.

**Explain:** Why does changing Age not increase the dictionary length?

Answer: Age already exists. Assignment replaces its value instead of adding a second Age key.



In [ ]:
# Predict before running: How many pairs remain after Campus is removed?
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
student["Age"] = 21
student["Program"] = "AI"
removed = student.pop("Campus")
print(student)
print("Removed:", removed)

**Your turn:** Start with the same original dictionary. Increase Age by 2, add Program as IT, and remove Campus. Print the final dictionary.

For the check, keep these result names: `student`.

**Hint:** Read the existing age on the right side of the assignment: student["Age"] + 2.

**Watch for this:** pop() raises KeyError for an absent key unless you also provide a default.

In [ ]:
# Change this code to complete the task.
# Question: Start with the same original dictionary. Increase Age by 2, add Program as IT, and remove Campus. Print the final dictionary.
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
student["Age"] = student["Age"] + 2
student["Program"] = "IT"
student.pop("Campus")
print(student)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert student == {'Name': 'Amira', 'Age': 22, 'Program': 'IT'}
print("Code checks passed. Review the written explanation as well.")

Age is 22 and Program is IT. Campus is no longer a key. Reading the old age lets the same calculation work for other starting ages.

**Quick check:** What happens when you assign another value to an existing key?

1. Its value is replaced.
2. A second identical key is added.
3. The dictionary is sorted.

Answer: 1. Each dictionary key identifies one entry; assigning to it replaces its associated value.

**Explain what you learned:** I can update a dictionary without confusing replacement with insertion.

### M01C04. Read all keys and all pairs

keys() gives the keys, values() gives their values, and items() gives key-value pairs. list() turns these views into lists that are easy to display. A for loop visits each pair in turn; the indented line runs once per pair.

**Read:** What two things are assigned to key and value on each loop turn?

Answer: One dictionary key and the value associated with that key.

**Predict:** How many times does the indented print line run?

Answer: Three times, once for each pair.

**Explain:** What is the difference between "RAK" in student and "RAK" in student.values()?

Answer: The first checks keys and is False. The second checks values and is True.



In [ ]:
# Predict before running: How many times does the indented print line run?
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
print(list(student.keys()))
print(list(student.values()))
for key, value in student.items():
    print(key, "=", value)

**Your turn:** Add Program as AI, print only the keys, and build a list named pairs using list(student.items()). Print pairs.

For the check, keep these result names: `pairs`.

**Hint:** items() exposes both parts of each dictionary entry together.

**Watch for this:** A Python dictionary preserves insertion order, but this does not mean the keys are alphabetically sorted.

In [ ]:
# Change this code to complete the task.
# Question: Add Program as AI, print only the keys, and build a list named pairs using list(student.items()). Print pairs.
student = {"Name": "Amira", "Age": 20, "Campus": "RAK", "Program": "AI"}
print(list(student.keys()))
pairs = list(student.items())
print(pairs)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert pairs == [('Name', 'Amira'), ('Age', 20), ('Campus', 'RAK'), ('Program', 'AI')]
print("Code checks passed. Review the written explanation as well.")

There are four keys and four pairs. Each pair is displayed as a tuple, such as ("Program", "AI").

**Quick check:** Which method gives key-value pairs?

1. values()
2. items()
3. keys()

Answer: 2. items() provides each key together with its value.

**Explain what you learned:** I can inspect a dictionary and explain what a key-value loop does.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates